# DiscoCH

In [ ]:
# User defined parameters
# Paths are machine-specific and come from config.yaml in the project root
# (copy config.example.yaml to config.yaml and set the paths for your machine).
from datetime import date
from src.disco_ch_operational.config import load_config

cfg = load_config()
network_directory = cfg["force_network_dir"]  # FORCE VI data on the network
force_directory = cfg["force_dir"]            # Local copy of the FORCE VI data
output_dir = cfg["output_dir"]                # Where to save output data
force_tiles_to_process = cfg["tiles_csv"]     # Tiles to process (Tile_ID column)
disco_model = cfg["model_path"]               # Pre-trained Discoloration Model
mask_path = cfg["mask_path"]                  # Forest mask (EPSG:2056, 1 = keep)
year = 2026

# The date up to which the code should be run: output dates are produced up to
# 5 days before this, each using data up to 5 days after it. date.today() in an
# operational setting when the code is run daily; never a date in the future.
date_to_run_to = date(year, 10, 5)

# Canonical VI name -> token used in the 2026 FORCE file names
vi_keys = {
    "NDVI": "NDVI",
    "EVI": "EVI",
    "NDMI": "NDMI",
    "CIRE": "CIre",
    "CCI": "CCI",
}

threshold = 4000  # Stored value (0-10000) at/above which a pixel is discolored: 4000 = 40 % probability


In [ ]:
# Libraries
import os
import matplotlib.pyplot as plt

from src.disco_ch_operational.transfer import transfer
from src.disco_ch_operational.interpolate import interpolate_tiles
from src.disco_ch_operational.apply_model import apply_discoch_tiles
from src.disco_ch_operational.mask import mask_and_scale_dir
from src.disco_ch_operational.validate import validate_threshold, validate_thresholds
from src.disco_ch_operational.disco_statistics import summarize_discoloration
from src.disco_ch_operational.compare import plot_year_comparison
from src.disco_ch_operational.hexmap import plot_hex_maps

# Reset to the standard light theme for plots
plt.style.use('default')

# Explicitly force figures to white:
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Output folders
force_output = os.path.join(output_dir, 'DiscoCH_rbf_10_20_30')
rbf_output = os.path.join(force_output, 'level4_rbf', str(year))     # interpolated VIs
discoCH_output = os.path.join(force_output, str(year))               # tiles, mosaic/, minmax/, ...
masked_output = os.path.join(discoCH_output, 'masked')
validation_output = os.path.join(discoCH_output, 'validation')
stats_output = os.path.join(discoCH_output, 'statistics')
full_data_output = os.path.join(force_output, f'{year}_full_data')   # written by DiscoCH_NRT_vs_Full_Data.ipynb


## Transfer data

In [ ]:
# Transfer VI data from the network to a local drive
transfer(network_directory, str(year), force_directory, dry_run=True)

## Interpolate

In [ ]:
os.makedirs(rbf_output, exist_ok=True)

results = interpolate_tiles(
    tss_dir=force_directory,
    out_dir=rbf_output,
    vi_keys=vi_keys,
    date_range=(date(year, 1, 1), date(year, 12, 31)),
    as_of=date_to_run_to,      # None = full season, no lag (e.g. training years)
    lag_days=5,
    int_day=5,
    rbf_sigma=(10, 20, 30), rbf_cutoff=0.95,
    despike=False,
    tiles_csv=force_tiles_to_process,  # only these Tile_IDs are processed
    skip_existing=True,        # keep dates already written, only add new ones
    chunk_size=512,
    max_workers=10,
)

for tile_id, by_vi in results.items():
    for vi, (tss_path, out_path, status) in by_vi.items():
        if status.startswith("failed"):
            print(f"{tile_id}/{vi}: {status}")

## Apply Discoloration Model

In [ ]:
# Forest mask (any extent/CRS -- each tile is matched to it automatically); None = all pixels
forest_mask = None

summary = apply_discoch_tiles(
    rbf_dir=rbf_output,
    out_dir=discoCH_output,          # tiles, mosaic/, minmax/, vi_norm/, minmax_by_date/, logs/
    year=year,
    model_path=disco_model,
    vi_keys=vi_keys,                 # same 2026 naming as the interpolation
    tiles_csv=force_tiles_to_process,
    forest_mask_path=forest_mask,
    start_date=f'{year}-01-01',      # min/max uses the season from here on
    model_months=(6, 7, 8, 9),       # model only runs in these months (min/max updates every date)
    export_normalized_vi=True,       # per-date normalized VIs -> vi_norm/
    export_minmax=True,              # per-date min/max snapshots -> minmax_by_date/
    export_start_date=f'{year}-06-01',
    export_end_date=f'{year}-09-30',
    mosaic=True,
    plot_mosaic=False,               # plot each newly mosaicked date
    max_workers=10,
)

## Masking

In [ ]:
# Mask the discoloration results and scale

status = mask_and_scale_dir(input_dir=os.path.join(discoCH_output, 'mosaic'),
                            mask_path=mask_path,  # None = only reproject to EPSG:2056 and scale
                            output_dir=masked_output,
                            pattern='Disco_Proba_*.tif',
                            threads=4,
                            )

## Validation

In [ ]:
# Calculate Validation statistics

validation_gpkg = cfg.require("validation_gpkg")        # Ground-truth polygons, Discolored = 1/0
validation_layer = "Sentinel_2_13_08_2026"                  # One layer per survey (also: Planet_2026_07_10)
proba_tif = os.path.join(masked_output, 'Disco_Proba_2026-08-14.tif')  # Mosaic closest to the survey date

# Sweep thresholds 0, 500, ..., 10000 (stored scale, 4000 = 40 %) to find the best F1
sweep = validate_thresholds(proba_tif, validation_gpkg, layer=validation_layer,
                            output_dir=validation_output)

# Score the best-F1 threshold in detail
validation_metrics = validate_threshold(proba_tif, validation_gpkg, sweep["f1"].idxmax(),
                                        layer=validation_layer, output_dir=validation_output)


## Spatiotemporal Statistics

In [ ]:
# Calculate national and district level statistics

districts = cfg.require("districts")  # Political districts (field bzname)

# Per date: discolored proportion and hectares for all of Switzerland ("All") and each district
stats = summarize_discoloration(masked_output, threshold, zones=districts, zone_field="bzname",
                                year=year, output_dir=stats_output, max_workers=8)


In [ ]:
# Compare the season with 2018 (Bloom et al. 2026 mosaics), aligned by month/day
mosaics_2018 = cfg.require("mosaics_2018")

comparison = plot_year_comparison(
    {
        "2018": dict(mosaic_dir=mosaics_2018, pattern="Proba_mosaic_*.tif", year=2018),
        str(year): stats,              # already counted above
    },
    threshold=threshold,
    output_dir=stats_output,
    max_workers=8,
)
# Per district: add zones=districts (so 2018 is counted by district too) and e.g.
# regions=["All", "Bezirk Aarau", "Distretto di Lugano"], layout="panels" (or "overlay").


## Hexagon Maps

In [ ]:
# End-of-month hexagon maps, 2018 vs this year (also saves hex_<year>_t<threshold>.gpkg with one field per date)
# Retrospective (all-data) run; use masked_output / stats_output for the near-real-time run
hex_masked = os.path.join(full_data_output, 'masked')
hex_stats = os.path.join(full_data_output, 'statistics')
districts = cfg.require("districts")
mosaics_2018 = cfg.require("mosaics_2018")

hexes = plot_hex_maps(
    {
        "2018": dict(mosaic_dir=mosaics_2018, pattern="Proba_mosaic_*.tif", year=2018),
        str(year): dict(mosaic_dir=hex_masked, year=year),
    },
    threshold=threshold,
    hex_area_km2=3,          # hexagon size
    boundary=districts,      # grid built over and clipped to Switzerland
    months=(6, 7, 8, 9),     # one column per month (its last mosaic date)
    output_dir=hex_stats,
    max_workers=8,
)


## Interpolation Examples

In [ ]:
# One pixel's observations and RBF interpolation: 2018 (retrospective) vs this year (retrospective and near-real-time)
from src.disco_ch_operational.visualize_interpolation import plot_pixel
from src.disco_ch_operational.compare import rc_params

force_directory_2018 = cfg.require("force_2018_dir")

tile, vi = "X0056_Y0061", "EVI"
row, col = 2425, 2156      # pixel within the tile (None, None = pick one with enough observations)

with plt.rc_context(rc_params(1.2)):
    fig, (ax18, ax26) = plt.subplots(2, 1, figsize=(15, 11))
plot_pixel(
    os.path.join(force_directory_2018, tile, f"2018-2018_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30, 50), rbf_cutoff=0.95, int_day=5,
    lag_days=None,         # retrospective ensemble only
    despike=True, above_noise=3.0, below_noise=1.0,
    title=f"2018, {tile} {vi}", ax=ax18, component_data="retrospective",
)
plot_pixel(
    os.path.join(force_directory, tile, f"{year}0101-{year}1231_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30), rbf_cutoff=0.95, int_day=5,
    lag_days=5, as_of=date_to_run_to,   # as in the Interpolate cell
    component_data="nrt",              # kernel curves near-real-time ("retrospective" = all data)
    despike=False,
    title=f"{year}, {tile} {vi}", ax=ax26,
)
# print_effective_bandwidth=True prints how much smoothing each date really got
fig.savefig(os.path.join(force_output, f"interpolation_{tile}_{vi}_r{row}_c{col}.png"), dpi=200, bbox_inches="tight")
plt.show()
